In [15]:
import pandas as pd
df=pd.read_parquet('all_airports.parquet')

In [16]:
df.columns

Index(['Carrier Code', 'Date (MM/DD/YYYY)', 'Flight Number', 'Tail Number',
       'Origin Airport', 'Destination Airport', 'Scheduled departure time',
       'Actual departure time', 'Scheduled elapsed time (Minutes)',
       'Actual elapsed time (Minutes)', 'Departure delay (Minutes)',
       'Arrival delay (Minutes)', 'Wheels-off time',
       'Taxi-Out time (Minutes)'],
      dtype='str')

In [17]:
df.head()

,Carrier Code,Date (MM/DD/YYYY),Flight Number,Tail Number,Origin Airport,Destination Airport,Scheduled departure time,Actual departure time,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),Departure delay (Minutes),Arrival delay (Minutes),Wheels-off time,Taxi-Out time (Minutes)
0,AS,01/01/2024,321,N972AK,ATL,SEA,18:02,18:12,343,309,10,-24,18:27,15
1,AS,01/01/2024,377,N531AS,ATL,SEA,07:00,07:44,341,312,44,15,07:58,14
2,AS,01/01/2024,497,N590AS,ATL,SEA,09:00,09:01,343,325,1,-17,09:23,22
3,AS,01/01/2025,321,N428AS,ATL,SEA,18:39,18:36,343,344,-3,-2,18:53,17
4,AS,01/01/2025,355,N588AS,ATL,SAN,17:19,17:16,284,297,-3,10,17:35,19


In [33]:
print(df.columns)
possible_columns = ['Carrier Code',
 'Date (MM/DD/YYYY)',
 'Flight Number',
 'Origin Airport',
 'Destination Airport',
 'Scheduled departure time',
 'Actual departure time',
 'Scheduled elapsed time (Minutes)']
complete_columns = [
    col for col in possible_columns
    if df[col].notna().all()
]
complete_columns

Index(['Carrier Code', 'Date (MM/DD/YYYY)', 'Flight Number', 'Tail Number',
       'Origin Airport', 'Destination Airport', 'Scheduled departure time',
       'Actual departure time', 'Scheduled elapsed time (Minutes)',
       'Actual elapsed time (Minutes)', 'Departure delay (Minutes)',
       'Arrival delay (Minutes)', 'Wheels-off time',
       'Taxi-Out time (Minutes)'],
      dtype='str')


['Carrier Code',
 'Date (MM/DD/YYYY)',
 'Flight Number',
 'Origin Airport',
 'Destination Airport',
 'Scheduled departure time',
 'Actual departure time',
 'Scheduled elapsed time (Minutes)']

In [39]:
from concurrent.futures import ThreadPoolExecutor
import itertools
import os

def find_composite_pks(df, max_combination_size=2, max_workers=4):
    complete_columns = [
        col for col in possible_columns
        if df[col].notna().all()
    ]

    combos = (
        combo
        for size in range(2, min(max_combination_size, len(complete_columns)) + 1)
        for combo in itertools.combinations(complete_columns, size)
    )

    def is_unique(combo):
        return combo if not df.duplicated(subset=list(combo)).any() else None

    with ThreadPoolExecutor(max_workers=max_workers or min(4, os.cpu_count() or 1)) as executor:
        results = executor.map(is_unique, combos)
        return [list(combo) for combo in results if combo is not None]

possible_pks = find_composite_pks(df, max_combination_size=5)
print("Possible Composite Primary Keys:", possible_pks)


Possible Composite Primary Keys: [['Carrier Code', 'Date (MM/DD/YYYY)', 'Flight Number', 'Origin Airport'], ['Carrier Code', 'Date (MM/DD/YYYY)', 'Flight Number', 'Scheduled departure time'], ['Carrier Code', 'Date (MM/DD/YYYY)', 'Flight Number', 'Origin Airport', 'Destination Airport'], ['Carrier Code', 'Date (MM/DD/YYYY)', 'Flight Number', 'Origin Airport', 'Scheduled departure time'], ['Carrier Code', 'Date (MM/DD/YYYY)', 'Flight Number', 'Origin Airport', 'Actual departure time'], ['Carrier Code', 'Date (MM/DD/YYYY)', 'Flight Number', 'Origin Airport', 'Scheduled elapsed time (Minutes)'], ['Carrier Code', 'Date (MM/DD/YYYY)', 'Flight Number', 'Destination Airport', 'Scheduled departure time'], ['Carrier Code', 'Date (MM/DD/YYYY)', 'Flight Number', 'Destination Airport', 'Actual departure time'], ['Carrier Code', 'Date (MM/DD/YYYY)', 'Flight Number', 'Destination Airport', 'Scheduled elapsed time (Minutes)'], ['Carrier Code', 'Date (MM/DD/YYYY)', 'Flight Number', 'Scheduled departur

In [38]:
df.shape

(3401526, 14)

In [ ]:
# Primary key - carrier_code, flight_number, date, origin_airport

In [27]:
route_groups = (
    df.groupby(
        ['Carrier Code', 'Origin Airport', 'Destination Airport'],
        dropna=False
    )
    .size()
    .reset_index(name='Flight Count')
)

route_groups.sample(n=min(20, len(route_groups)), random_state=42)
flight_groups = df.groupby(
    ['Carrier Code', 'Origin Airport', 'Destination Airport'],
    dropna=False
)

flight_groups.get_group(('AA', 'ATL', 'DFW')).head(20)

,Carrier Code,Date (MM/DD/YYYY),Flight Number,Tail Number,Origin Airport,Destination Airport,Scheduled departure time,Actual departure time,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),Departure delay (Minutes),Arrival delay (Minutes),Wheels-off time,Taxi-Out time (Minutes)
2975,AA,01/01/2024,1236,N128AN,ATL,DFW,16:24,17:43,155,129,79,53,17:58,15
2978,AA,01/01/2024,1248,N902AA,ATL,DFW,18:19,18:10,153,145,-9,-17,18:26,16
2980,AA,01/01/2024,1276,N534UW,ATL,DFW,12:16,12:03,156,137,-13,-32,12:16,13
2981,AA,01/01/2024,1567,N934AA,ATL,DFW,07:21,07:11,156,146,-10,-20,07:25,14
2987,AA,01/01/2024,2403,N907AA,ATL,DFW,20:16,20:32,148,135,16,3,20:43,11
2988,AA,01/01/2024,2476,N991AN,ATL,DFW,11:16,11:12,167,177,-4,6,11:42,30
2989,AA,01/01/2024,2770,N140AN,ATL,DFW,05:47,05:37,153,177,-10,14,06:03,26
2999,AA,01/01/2025,1224,N973AN,ATL,DFW,12:19,12:10,160,153,-9,-16,12:28,18
3000,AA,01/01/2025,1236,N844NN,ATL,DFW,14:10,14:05,160,152,-5,-13,14:23,18
3003,AA,01/01/2025,1248,N9002U,ATL,DFW,18:16,18:10,153,170,-6,11,18:27,17


In [24]:
aa_flights = df[df['Carrier Code'] == 'AA']

print(f"Total AA flights: {len(aa_flights)}")
print(f"Unique AA tail numbers: {aa_flights['Tail Number'].nunique()}")

Total AA flights: 789387
Unique AA tail numbers: 1013


In [40]:
df.shape

(3401526, 14)

In [45]:
states=pd.read_excel(r"C:\Users\Nishad\Downloads\50States.xlsx")
states

states.set_index('Abbr')['State'].to_json('states.json')

In [52]:
df.head()

,Carrier Code,Date (MM/DD/YYYY),Flight Number,Tail Number,Origin Airport,Destination Airport,Scheduled departure time,Actual departure time,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),Departure delay (Minutes),Arrival delay (Minutes),Wheels-off time,Taxi-Out time (Minutes)
0,AS,01/01/2024,321,N972AK,ATL,SEA,18:02,18:12,343,309,10,-24,18:27,15
1,AS,01/01/2024,377,N531AS,ATL,SEA,07:00,07:44,341,312,44,15,07:58,14
2,AS,01/01/2024,497,N590AS,ATL,SEA,09:00,09:01,343,325,1,-17,09:23,22
3,AS,01/01/2025,321,N428AS,ATL,SEA,18:39,18:36,343,344,-3,-2,18:53,17
4,AS,01/01/2025,355,N588AS,ATL,SAN,17:19,17:16,284,297,-3,10,17:35,19


contruct an airline table with columns (carrier code and airline name) (pick these details from carriers.json), also construct the airports table using a union of all the unique airport codes from origin airport and destination airport column, (for the airports table the column names are airport code , airport name, city, state  ) for airports table you will have to do a bit more, in airports.json the keys are the airport codes and the values are the full details like this "Hartsfield International, Atlanta, GA" where thr first item is the airport name, then its city and then state, but for state its abbreviations, so you will have to pick the full state names from states.json where the keys are the abbreviations and put the full name in that table

In [59]:
import json

# Load the carrier-code-to-airline-name mapping.
with open("carriers.json", encoding="utf-8") as file:
	carriers = json.load(file)

airline_table = pd.DataFrame(
	[{"Carrier Code": code, "Airline Name": name} for code, name in carriers.items()]
)

# Map state abbreviations to full names using the existing states DataFrame.
state_names = states.set_index("Abbr")["State"].to_dict()

with open("airports.json", encoding="utf-8") as file:
	airport_details = json.load(file)

airport_codes = sorted(
	set(df["Origin Airport"].dropna().unique())
	| set(df["Destination Airport"].dropna().unique())
)

airport_rows = []
for code in airport_codes:
	details = airport_details.get(code, "")
	parts = [part.strip() for part in details.split(",")]

	airport_name = parts[0] if len(parts) > 0 else None
	city = parts[1] if len(parts) > 1 else None
	state_abbr = parts[2] if len(parts) > 2 else None

	airport_rows.append({
		"Airport Code": code,
		"Airport Name": airport_name,
		"City": city,
		"State": state_names.get(state_abbr, state_abbr),
	})

airports_table = pd.DataFrame(
	airport_rows,
	columns=["Airport Code", "Airport Name", "City", "State"]
)

airline_table

,Carrier Code,Airline Name
0,AA,American Airlines
1,AQ,Aloha Airlines
2,AS,Alaska Airlines
3,CO,Continental Airlines
4,DL,Delta Air Lines
5,HP,America West Airlines
6,MQ,American Eagle Airlines
7,NW,Northwest Airlines
8,TW,Trans World Airlines
9,UA,United Airlines


In [60]:
airports_table

,Airport Code,Airport Name,City,State
0,ABQ,Albuquerque International,Albuquerque,New Mexico
1,ACK,Nantucket Memorial,Nantucket,Massachusetts
2,AGS,Augusta Regional,Augusta,Georgia
3,ALB,Albany County,Albany,New York
4,AMA,Amarillo International,Amarillo,Texas
...,...,...,...,...
175,TVC,Cherry Capital,Traverse City,Michigan
176,TYS,McGhee Tyson,Knoxville,Tennessee
177,VPS,Eglin Air Force Base,Valparaiso,Florida
178,VRB,Vero Beach Municipal,Vero Beach,Florida


In [63]:
print(airline_table.columns)
print(airports_table.columns)
print(df.columns)

Index(['Carrier Code', 'Airline Name'], dtype='str')
Index(['Airport Code', 'Airport Name', 'City', 'State'], dtype='str')
Index(['Carrier Code', 'Date (MM/DD/YYYY)', 'Flight Number', 'Tail Number',
       'Origin Airport', 'Destination Airport', 'Scheduled departure time',
       'Actual departure time', 'Scheduled elapsed time (Minutes)',
       'Actual elapsed time (Minutes)', 'Departure delay (Minutes)',
       'Arrival delay (Minutes)', 'Wheels-off time',
       'Taxi-Out time (Minutes)'],
      dtype='str')


##### Prepare the dataframes

First, we will standardize the column names, clean identifiers, convert the dates and times, and validate the composite primary key (# Primary key - carrier_code, flight_number, date, origin_airport) before touching the database.

Note: the handling of 24:00 above maps it to 00:00. Since SQL Server's TIME type cannot represent 24:00 or a day rollover, verify any such records before using the stored times for date-sensitive calculations.

In [64]:
import pandas as pd
import numpy as np
import pyodbc

airlines = airline_table.rename(columns={
    "Carrier Code": "carrier_code",
    "Airline Name": "airline_name"
}).copy()

airports = airports_table.rename(columns={
    "Airport Code": "airport_code",
    "Airport Name": "airport_name",
    "City": "city",
    "State": "state"
}).copy()

flights = df.rename(columns={
    "Carrier Code": "carrier_code",
    "Date (MM/DD/YYYY)": "flight_date",
    "Flight Number": "flight_number",
    "Tail Number": "tail_number",
    "Origin Airport": "origin_airport_code",
    "Destination Airport": "destination_airport_code",
    "Scheduled departure time": "scheduled_departure_time",
    "Actual departure time": "actual_departure_time",
    "Scheduled elapsed time (Minutes)": "scheduled_elapsed_minutes",
    "Actual elapsed time (Minutes)": "actual_elapsed_minutes",
    "Departure delay (Minutes)": "departure_delay_minutes",
    "Arrival delay (Minutes)": "arrival_delay_minutes",
    "Wheels-off time": "wheels_off_time",
    "Taxi-Out time (Minutes)": "taxi_out_minutes"
}).copy()


# Standardize codes and identifiers.
def clean_text(series, uppercase=False):
    result = series.astype("string").str.strip()

    if uppercase:
        result = result.str.upper()

    return result.replace("", pd.NA)


airlines["carrier_code"] = clean_text(
    airlines["carrier_code"], uppercase=True
)
airports["airport_code"] = clean_text(
    airports["airport_code"], uppercase=True
)

flights["carrier_code"] = clean_text(
    flights["carrier_code"], uppercase=True
)
flights["origin_airport_code"] = clean_text(
    flights["origin_airport_code"], uppercase=True
)
flights["destination_airport_code"] = clean_text(
    flights["destination_airport_code"], uppercase=True
)
flights["tail_number"] = clean_text(
    flights["tail_number"], uppercase=True
)

# Keep flight numbers as strings, avoiding unwanted ".0" suffixes
# if pandas originally inferred a numeric column.
flights["flight_number"] = (
    clean_text(flights["flight_number"])
    .str.replace(r"\.0$", "", regex=True)
)

for col in ["airline_name"]:
    airlines[col] = clean_text(airlines[col])

for col in ["airport_name", "city", "state"]:
    airports[col] = clean_text(airports[col])


# Parse flight dates. The source is MM/DD/YYYY.
date_text = flights["flight_date"].astype("string").str.strip()

parsed_dates = pd.to_datetime(
    date_text,
    format="%m/%d/%Y",
    errors="coerce"
)

# Also accommodate dates already represented in ISO format.
fallback_mask = parsed_dates.isna() & date_text.notna()

if fallback_mask.any():
    parsed_dates.loc[fallback_mask] = pd.to_datetime(
        date_text.loc[fallback_mask],
        errors="coerce"
    )

flights["flight_date"] = parsed_dates.dt.date


# Parse time values such as "18:02", "18:02:00", or "1802".
def parse_time_column(series):
    text = series.astype("string").str.strip()
    text = text.str.replace(r"\.0$", "", regex=True)

    text = text.replace({
        "": pd.NA,
        "nan": pd.NA,
        "NaN": pd.NA,
        "None": pd.NA,
        "24:00": "00:00",
        "24:00:00": "00:00:00",
        "2400": "0000"
    })

    digit_mask = text.str.fullmatch(r"\d{1,4}").fillna(False)
    text.loc[digit_mask] = text.loc[digit_mask].str.zfill(4)

    parsed = pd.to_datetime(
        text, format="%H:%M", errors="coerce"
    )

    parsed = parsed.fillna(
        pd.to_datetime(text, format="%H:%M:%S", errors="coerce")
    )

    parsed = parsed.fillna(
        pd.to_datetime(text, format="%H%M", errors="coerce")
    )

    # Reject unexpected non-empty values instead of silently
    # loading them as NULL.
    invalid = text.notna() & parsed.isna()

    if invalid.any():
        examples = text.loc[invalid].head(10).tolist()
        raise ValueError(
            f"Unrecognized time values in {series.name}: {examples}"
        )

    return parsed.dt.time


time_columns = [
    "scheduled_departure_time",
    "actual_departure_time",
    "wheels_off_time"
]

for col in time_columns:
    flights[col] = parse_time_column(flights[col])


# Convert measurements to nullable integers.
integer_columns = [
    "scheduled_elapsed_minutes",
    "actual_elapsed_minutes",
    "departure_delay_minutes",
    "arrival_delay_minutes",
    "taxi_out_minutes"
]

for col in integer_columns:
    source = flights[col].replace(
        ["", "-", "N/A", "NA"], pd.NA
    )

    numeric = pd.to_numeric(source, errors="coerce")

    invalid = source.notna() & numeric.isna()

    if invalid.any():
        examples = source.loc[invalid].head(10).tolist()
        raise ValueError(
            f"Invalid numeric values in {col}: {examples}"
        )

    fractional = numeric.notna() & (numeric % 1 != 0)

    if fractional.any():
        raise ValueError(
            f"{col} contains non-integer values."
        )

    flights[col] = numeric.astype("Int64")

In [70]:
#verifying pk and fk constraints

# Remove no data silently: reject missing or duplicate airline keys.
airlines = airlines.dropna(subset=["carrier_code"])

if airlines["carrier_code"].duplicated().any():
    raise ValueError("Duplicate carrier codes in airline_table.")

# Every carrier in a flight must exist in Airline.
missing_carriers = (
    set(flights["carrier_code"].dropna())
    - set(airlines["carrier_code"])
)

if missing_carriers:
    raise ValueError(
        f"Missing airline records for: {sorted(missing_carriers)}"
    )

# Check that required flight fields are present.
primary_key = [
    "carrier_code",
    "flight_date",
    "flight_number",
    "origin_airport_code"
]

required_flight_fields = primary_key + [
    "destination_airport_code"
]

null_counts = flights[required_flight_fields].isna().sum()

if null_counts.any():
    raise ValueError(
        "Missing required flight identifiers:\n"
        + null_counts[null_counts > 0].to_string()
    )

# SQL Server CHAR(3) is used for airport codes.
for col in ["origin_airport_code", "destination_airport_code"]:
    invalid_length = flights[col].str.len().ne(3)

    if invalid_length.any():
        raise ValueError(
            f"{col} contains airport codes that are not 3 characters."
        )

# Validate the four-column composite key AFTER normalizing the data.
duplicate_mask = flights.duplicated(
    subset=primary_key,
    keep=False
)

if duplicate_mask.any():
    raise ValueError(
        f"The proposed primary key is not unique after cleaning. "
        f"{duplicate_mask.sum():,} rows participate in duplicate keys.\n"
        f"{flights.loc[duplicate_mask, primary_key].head(10)}"
    )

print("Composite primary-key validation passed.")
print("Flight rows to load:", f"{len(flights):,}")

Composite primary-key validation passed.
Flight rows to load: 3,401,526


##### DB and Tables Creation

In [61]:
import pyodbc
import pandas as pd

def run_query(query: str, database: str = "master") -> pd.DataFrame:
    """
    All-in-one function to execute ANY T-SQL command (CREATE, INSERT, SELECT, etc.)
    against your local Microsoft SQL Server instance.
    """
    conn_str = (
        r"Driver={ODBC Driver 18 for SQL Server};"
        f"Server=DESKTOP-LVP84RD\MSSQLSERVER01;"
        f"Database={database};"
        r"Trusted_Connection=yes;"
        r"TrustServerCertificate=yes;"
    )
    
    conn = None
    try:
        # Open connection and enable autocommit so structural changes save instantly
        conn = pyodbc.connect(conn_str, autocommit=True)
        cursor = conn.cursor()
        
        # Execute the raw query string
        cursor.execute(query)
        
        # Check if the query returned rows (like a SELECT statement does)
        if cursor.description:
            # Fetch the metadata columns and rows into a structured DataFrame
            columns = [column[0] for column in cursor.description]
            rows = cursor.fetchall()
            # Convert pyodbc rows to standard python lists for Pandas compatibility
            data = [list(row) for row in rows]
            return pd.DataFrame(data, columns=columns)
        else:
            print("🚀 Command executed successfully (No rows returned).")
            return None
            
    except Exception as e:
        print(f"❌ Execution Error: {e}")
        return None
        
    finally:
        if conn:
            conn.close()

<>:11: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
<>:11: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
C:\Users\Nishad\AppData\Local\Temp\ipykernel_20892\2771818840.py:11: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
  f"Server=DESKTOP-LVP84RD\MSSQLSERVER01;"


In [50]:
# print("--- 1. Creating the Database ---")
# run_query("CREATE DATABASE AirlinesProject;")


--- 1. Creating the Database ---
🚀 Command executed successfully (No rows returned).


In [51]:
# Query the system database catalog
databases_df = run_query("SELECT name AS DatabaseName, create_date FROM sys.databases;")

# Display the table
databases_df


,DatabaseName,create_date
0,master,2003-04-08 09:13:36.390
1,tempdb,2026-11-09 20:15:34.737
2,model,2003-04-08 09:13:36.390
3,msdb,2025-10-21 12:53:13.233
4,LearnSQL,2026-09-21 18:21:46.770
5,AirlinesProject,2026-11-10 12:40:22.793


In [72]:
# Query the system table catalog for your specific database
tables_df = run_query("SELECT name AS TableName, create_date FROM sys.tables;", database="AirlinesProject")

# Display the table
tables_df


,TableName,create_date
0,Airline,2026-11-10 13:12:54.100
1,Airport,2026-11-10 13:12:54.127
2,FlightInstance,2026-11-10 13:12:54.133


In [71]:
# create Airline table
run_query("""
CREATE TABLE dbo.Airline (
    carrier_code VARCHAR(5) NOT NULL,
    airline_name VARCHAR(100) NULL,

    CONSTRAINT PK_Airline
        PRIMARY KEY (carrier_code)
);
""", database="AirlinesProject")

# create Airport table
run_query("""
CREATE TABLE dbo.Airport (
    airport_code CHAR(3) NOT NULL,
    airport_name VARCHAR(150) NULL,
    city VARCHAR(100) NULL,
    state VARCHAR(50) NULL,

    CONSTRAINT PK_Airport
        PRIMARY KEY (airport_code)
);
""", database="AirlinesProject")

#create FlightInstance table
run_query("""
CREATE TABLE dbo.FlightInstance (
    carrier_code VARCHAR(5) NOT NULL,
    flight_date DATE NOT NULL,
    flight_number VARCHAR(10) NOT NULL,
    tail_number VARCHAR(10) NULL,

    origin_airport_code CHAR(3) NOT NULL,
    destination_airport_code CHAR(3) NOT NULL,

    scheduled_departure_time TIME(0) NULL,
    actual_departure_time TIME(0) NULL,
    scheduled_elapsed_minutes INT NULL,
    actual_elapsed_minutes INT NULL,
    departure_delay_minutes INT NULL,
    arrival_delay_minutes INT NULL,
    wheels_off_time TIME(0) NULL,
    taxi_out_minutes INT NULL,

    CONSTRAINT PK_FlightInstance
        PRIMARY KEY CLUSTERED (
            carrier_code,
            flight_date,
            flight_number,
            origin_airport_code
        ),

    CONSTRAINT FK_FlightInstance_Airline
        FOREIGN KEY (carrier_code)
        REFERENCES dbo.Airline(carrier_code),

    CONSTRAINT FK_FlightInstance_OriginAirport
        FOREIGN KEY (origin_airport_code)
        REFERENCES dbo.Airport(airport_code),

    CONSTRAINT FK_FlightInstance_DestinationAirport
        FOREIGN KEY (destination_airport_code)
        REFERENCES dbo.Airport(airport_code)
);
""", database="AirlinesProject")

🚀 Command executed successfully (No rows returned).
🚀 Command executed successfully (No rows returned).
🚀 Command executed successfully (No rows returned).


In SQL Server, dbo. is the schema name. It tells SQL Server which schema a table belongs to.
For example:
SELECT * FROM dbo.FlightInstance;

Here:
- dbo = schema name
- FlightInstance = table name
Think of a schema as a namespace or folder inside a database that organizes database objects such as tables, views, and stored procedures.

dbo stands for database owner. It is the conventional default schema in SQL Server for many database users, though it is technically a schema, not a database or a folder.

In [74]:
tables_df = run_query("""
SELECT
    TABLE_SCHEMA,
    TABLE_NAME
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_TYPE = 'BASE TABLE'
ORDER BY TABLE_NAME;
""", database="AirlinesProject")

display(tables_df)

,TABLE_SCHEMA,TABLE_NAME
0,dbo,Airline
1,dbo,Airport
2,dbo,FlightInstance


In [75]:
columns_df = run_query("""
SELECT
    TABLE_NAME,
    COLUMN_NAME,
    DATA_TYPE,
    IS_NULLABLE
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'dbo'
ORDER BY TABLE_NAME, ORDINAL_POSITION;
""", database="AirlinesProject")

display(columns_df)

,TABLE_NAME,COLUMN_NAME,DATA_TYPE,IS_NULLABLE
0,Airline,carrier_code,varchar,NO
1,Airline,airline_name,varchar,YES
2,Airport,airport_code,char,NO
3,Airport,airport_name,varchar,YES
4,Airport,city,varchar,YES
5,Airport,state,varchar,YES
6,FlightInstance,carrier_code,varchar,NO
7,FlightInstance,flight_date,date,NO
8,FlightInstance,flight_number,varchar,NO
9,FlightInstance,tail_number,varchar,YES


In [76]:
# inserting values into the tables, the order of insertion is important because of the foreign key constraints. First, we insert into the Airline table, then the Airport table, and finally the FlightInstance table. because the FlightInstance table has foreign key constraints that reference the Airline and Airport tables. If we try to insert data into the FlightInstance table before the referenced records exist in the Airline and Airport tables, it will violate the foreign key constraints and result in an error. Therefore, we must ensure that the referenced records are present in the Airline and Airport tables before inserting data into the FlightInstance table.

In [77]:
#For 3.4 million rows, don't insert records one at a time. Use pyodbc's fast_executemany feature and insert batches.
#run_query() function is useful for running SQL commands, but unless it accepts parameterized batches, it isn't the right tool for transferring millions of DataFrame records. We'll use a dedicated loading function below.

In [78]:
#connect
import pyodbc
import pandas as pd
import numpy as np

conn_str = (
    r"Driver={ODBC Driver 18 for SQL Server};"
    r"Server=DESKTOP-LVP84RD\MSSQLSERVER01;"
    r"Database=AirlinesProject;"
    r"Trusted_Connection=yes;"
    r"TrustServerCertificate=yes;"
)

conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

cursor.fast_executemany = True
#############
#reusable batch-insertion function - converts pandas missing values to SQL NULL, prepares parameterized inserts, and commits every batch.

def to_db_value(value):
    # Convert missing pandas/numpy values to SQL NULL.
    if value is None or pd.isna(value):
        return None

    # Convert numpy scalars to standard Python values.
    if isinstance(value, np.generic):
        return value.item()

    # Convert pandas timestamps to Python datetime objects.
    if isinstance(value, pd.Timestamp):
        return value.to_pydatetime()

    return value

def insert_dataframe(
    cursor,
    conn,
    table_name,
    data,
    columns,
    batch_size=20000
):
    column_sql = ", ".join(f"[{c}]" for c in columns)
    placeholders = ", ".join("?" for _ in columns)

    sql = (
        f"INSERT INTO dbo.{table_name} ({column_sql}) "
        f"VALUES ({placeholders})"
    )

    total = len(data)

    for start in range(0, total, batch_size):
        end = min(start + batch_size, total)

        batch = data.iloc[start:end][columns]

        records = [
            tuple(to_db_value(value) for value in row)
            for row in batch.itertuples(index=False, name=None)
        ]

        try:
            cursor.executemany(sql, records)
            conn.commit()
        except Exception:
            conn.rollback()
            raise

        if end == total or (start // batch_size) % 10 == 0:
            print(f"{table_name}: {end:,} / {total:,} rows inserted")

    print(f"Finished loading {table_name}: {total:,} rows.")

############
 #making sure tables are empty so that we dont accidentaly load the same data twice

for table in ["Airline", "Airport", "FlightInstance"]:
    count = cursor.execute(
        f"SELECT COUNT_BIG(*) FROM dbo.{table}"
    ).fetchone()[0]

    print(f"{table}: {count:,} existing rows")

    if count != 0:
        raise RuntimeError(
            f"{table} is not empty. Stop before loading to avoid duplicates."
        )

Airline: 0 existing rows
Airport: 0 existing rows
FlightInstance: 0 existing rows


In [79]:
#loading parent tables airlines and flight
import time
start=time.perf_counter()
insert_dataframe(
    cursor,
    conn,
    "Airline",
    airlines,
    ["carrier_code", "airline_name"],
    batch_size=1000
)

insert_dataframe(
    cursor,
    conn,
    "Airport",
    airports,
    ["airport_code", "airport_name", "city", "state"],
    batch_size=1000
)
end=time.perf_counter()

print("Time taken to load airlines and airports:", end-start, "seconds")

Airline: 14 / 14 rows inserted
Finished loading Airline: 14 rows.
Airport: 180 / 180 rows inserted
Finished loading Airport: 180 rows.
Time taken to load airlines and airports: 0.09536409999418538 seconds


In [80]:
#3.4 million flight records
start=time.perf_counter()
flight_columns = [
    "carrier_code",
    "flight_date",
    "flight_number",
    "tail_number",
    "origin_airport_code",
    "destination_airport_code",
    "scheduled_departure_time",
    "actual_departure_time",
    "scheduled_elapsed_minutes",
    "actual_elapsed_minutes",
    "departure_delay_minutes",
    "arrival_delay_minutes",
    "wheels_off_time",
    "taxi_out_minutes"
]

insert_dataframe(
    cursor,
    conn,
    "FlightInstance",
    flights,
    flight_columns,
    batch_size=20000
)
end=time.perf_counter()
print("Time taken to load airlines and airports:", end-start, "seconds")

#This loads all 14 original flight columns in batches of 20,000 rows, rather than making 3.4 million individual insert calls.
#Important: Each batch is committed independently. If an error occurs midway through loading, the previously committed batches will remain in SQL Server. Don't blindly rerun the entire load; inspect the row counts and clear the partial load in the appropriate order before restarting.

FlightInstance: 20,000 / 3,401,526 rows inserted
FlightInstance: 220,000 / 3,401,526 rows inserted
FlightInstance: 420,000 / 3,401,526 rows inserted
FlightInstance: 620,000 / 3,401,526 rows inserted
FlightInstance: 820,000 / 3,401,526 rows inserted
FlightInstance: 1,020,000 / 3,401,526 rows inserted
FlightInstance: 1,220,000 / 3,401,526 rows inserted
FlightInstance: 1,420,000 / 3,401,526 rows inserted
FlightInstance: 1,620,000 / 3,401,526 rows inserted
FlightInstance: 1,820,000 / 3,401,526 rows inserted
FlightInstance: 2,020,000 / 3,401,526 rows inserted
FlightInstance: 2,220,000 / 3,401,526 rows inserted
FlightInstance: 2,420,000 / 3,401,526 rows inserted
FlightInstance: 2,620,000 / 3,401,526 rows inserted
FlightInstance: 2,820,000 / 3,401,526 rows inserted
FlightInstance: 3,020,000 / 3,401,526 rows inserted
FlightInstance: 3,220,000 / 3,401,526 rows inserted
FlightInstance: 3,401,526 / 3,401,526 rows inserted
Finished loading FlightInstance: 3,401,526 rows.
Time taken to load airline

In [81]:
cursor.close()
conn.close()

#### Setup done, now verifying and inspection

In [85]:
counts_df = run_query("""
SELECT 'Airline' AS TableName, COUNT_BIG(*) AS TotalRows
FROM Airline

UNION ALL

SELECT 'Airport', COUNT_BIG(*)
FROM Airport

UNION ALL

SELECT 'FlightInstance', COUNT_BIG(*)
FROM dbo.FlightInstance;
""", database="AirlinesProject")

display(counts_df)

,TableName,TotalRows
0,Airline,14
1,Airport,180
2,FlightInstance,3401526


In [86]:
print("Expected Airline rows:", len(airlines))
print("Expected Airport rows:", len(airports))
print("Expected FlightInstance rows:", len(flights))

Expected Airline rows: 14
Expected Airport rows: 180
Expected FlightInstance rows: 3401526


In [88]:
#joining table on the FK contrain of carrier code and airport to print the full names
sample_df = run_query("""
SELECT TOP (10)
    f.carrier_code,
    a.airline_name,
    f.flight_date,
    f.flight_number,
    f.origin_airport_code,
    origin.airport_name AS origin_airport_name,
    f.destination_airport_code,
    destination.airport_name AS destination_airport_name,
    f.scheduled_departure_time,
    f.actual_departure_time,
    f.departure_delay_minutes,
    f.arrival_delay_minutes
FROM dbo.FlightInstance AS f
JOIN dbo.Airline AS a
    ON f.carrier_code = a.carrier_code
JOIN dbo.Airport AS origin
    ON f.origin_airport_code = origin.airport_code
JOIN dbo.Airport AS destination
    ON f.destination_airport_code = destination.airport_code
ORDER BY
    f.flight_date,
    f.carrier_code,
    f.flight_number;
""", database="AirlinesProject")

display(sample_df)

#This demonstrates that your normalized tables are connected correctly and that you can retrieve airline and airport details using foreign-key joins.

,carrier_code,airline_name,flight_date,flight_number,origin_airport_code,origin_airport_name,destination_airport_code,destination_airport_name,scheduled_departure_time,actual_departure_time,departure_delay_minutes,arrival_delay_minutes
0,AA,American Airlines,2024-01-01,1010,DFW,Dallas/Ft.Worth International,STL,Lambert-St Louis International,21:09:00,21:34:00,25,12
1,AA,American Airlines,2024-01-01,1011,LAS,McCarran International,CLT,Charlotte Douglas International,08:00:00,07:55:00,-5,-14
2,AA,American Airlines,2024-01-01,1021,DFW,Dallas/Ft.Worth International,GSO,Piedmont Triad International,19:11:00,19:06:00,-5,-5
3,AA,American Airlines,2024-01-01,1022,DFW,Dallas/Ft.Worth International,SAT,San Antonio International,15:41:00,15:54:00,13,9
4,AA,American Airlines,2024-01-01,1023,DFW,Dallas/Ft.Worth International,LAX,Los Angeles International,17:31:00,17:29:00,-2,-17
5,AA,American Airlines,2024-01-01,1032,JFK,John F Kennedy International,MIA,Miami International,09:00:00,09:01:00,1,-9
6,AA,American Airlines,2024-01-01,1036,LAX,Los Angeles International,STL,Lambert-St Louis International,18:20:00,18:17:00,-3,0
7,AA,American Airlines,2024-01-01,1045,MIA,Miami International,ATL,Hartsfield International,19:02:00,20:42:00,100,93
8,AA,American Airlines,2024-01-01,1046,MIA,Miami International,ATL,Hartsfield International,08:41:00,08:40:00,-1,-9
9,AA,American Airlines,2024-01-01,1048,DFW,Dallas/Ft.Worth International,BOS,Logan International,15:45:00,16:07:00,22,40


In [91]:
tables_with_sizes = run_query("""
SELECT 
    t.TABLE_SCHEMA, 
    t.TABLE_NAME,
    SUM(p.row_count) AS [RowCount],
    CAST(ROUND(((SUM(p.used_page_count) * 8.0) / 1024.0), 2) AS DECIMAL(36, 2)) AS [TotalSpace_MB],
    CAST(ROUND(((SUM(p.used_page_count) * 8.0) / 1024.0 / 1024.0), 2) AS DECIMAL(36, 2)) AS [TotalSpace_GB],
    CAST(ROUND(((SUM(CASE WHEN p.index_id < 2 THEN p.in_row_data_page_count + p.lob_used_page_count + p.row_overflow_used_page_count ELSE 0 END) * 8.0) / 1024.0), 2) AS DECIMAL(36, 2)) AS [DataSpace_MB],
    CAST(ROUND(((SUM(p.used_page_count - CASE WHEN p.index_id < 2 THEN p.in_row_data_page_count + p.lob_used_page_count + p.row_overflow_used_page_count ELSE 0 END) * 8.0) / 1024.0), 2) AS DECIMAL(36, 2)) AS [IndexSpace_MB]
FROM INFORMATION_SCHEMA.TABLES t
JOIN sys.tables st 
    ON t.TABLE_NAME = st.name 
    AND t.TABLE_SCHEMA = SCHEMA_NAME(st.schema_id)
JOIN sys.dm_db_partition_stats p 
    ON st.object_id = p.object_id
WHERE t.TABLE_TYPE = 'BASE TABLE'
GROUP BY t.TABLE_SCHEMA, t.TABLE_NAME
ORDER BY [TotalSpace_MB] DESC;
""", database="AirlinesProject")

display(tables_with_sizes)


,TABLE_SCHEMA,TABLE_NAME,RowCount,TotalSpace_MB,TotalSpace_GB,DataSpace_MB,IndexSpace_MB
0,dbo,FlightInstance,3401526,346.69,0.34,345.12,1.57
1,dbo,Airport,180,0.03,0.00,0.02,0.02
2,dbo,Airline,14,0.02,0.00,0.01,0.01
